
# Scopes APTs for all of the following
1. `feedaback:*` tag
      - feedback:feature
      - feedback:geometry
      - feedback:property:*
______________


## Pre-requisite:
### Refresh RP snapshot before triggering the scoping

link - https://adb-2440482787921388.8.azuredatabricks.net/editor/notebooks/2055378621412004?o=2440482787921388#command/2055378621412005
__

In [0]:
import org.apache.spark.sql.functions._
import org.apache.commons.lang.StringUtils

// UDF to convert signed Long to unsigned string
val toUnsignedLong = udf((signedStr: String) => {
  if (signedStr == null || signedStr.isEmpty) {
    "0"
  } else {
    try {
      val signedLong = signedStr.toLong
      java.lang.Long.toUnsignedString(signedLong)
    } catch {
      case _: NumberFormatException => signedStr // return as-is if not numeric
    }
  }
})



### Load APT data 


In [0]:
val aptRawTable = spark.read.format("delta").table("delete_retriggers.layer_14533")

val scopedApts = aptRawTable
  .drop("revisionId", "elementType", "wkt", "members", "semanticIds", "nodes")
  .withColumn(
    "Product_orbis_id",
    concat(
      col("id.layerId"), lit(":"),
      toUnsignedLong(col("id.high")), lit(":"),
      toUnsignedLong(col("id.low"))
    )
  )
  .withColumn(
    "countryIso",
    expr("filter(tags, x -> x.tagKey.key == 'metadata:country')[0].value")
  )
  .withColumn("tagKeyValue", explode(col("tags")))
  .filter(col("tagKeyValue.tagKey.key").startsWith("feedback:"))
  .repartition(col("Product_orbis_id"))
  .groupBy(col("Product_orbis_id"))
  .agg(
      first("countryIso").as("countryIso"),
      collect_list(col("tagKeyValue")).as("feedaback_tags")
    )
    
  

// scopedApts.show(20, false)
// println("count: "+ scopedApts.count())

// aptIdWithFeedbackTags :26794

# Invoke this to process the data
- Build Changelet 
- Upload to BlobStore

In [0]:
import com.tomtom.orbis.NodeGeometry
import com.tomtom.orbis.Id

def convertStringToUnsignedLong(value: String): Long = {
  java.lang.Long.parseUnsignedLong(value)
}

def createOrbisIdFromString(id: String): Id = {
  val split = id.split(":")
  Id.newBuilder()
    .setLayerId(split(0).toInt)
    .setHigh(convertStringToUnsignedLong(split(1)))
    .setLow(convertStringToUnsignedLong(split(2)))
    .build()
}

def createOrbisIdFromStringInSpark(id: String): com.tomtom.orbis.io.spark.model.Id = {
  val split = id.split(":")
  new com.tomtom.orbis.io.spark.model.Id(layerId = Option.apply(split(0).toInt), high = convertStringToUnsignedLong(split(1)), low = convertStringToUnsignedLong(split(2)))
}



// Constants for feedback tag values
val ADDRESS_POINT_FEEDBACK_FEATURE  = "1271990176"
val ADDRESS_POINT_FEEDBACK_GEOMETRY = "2913669384"
val ADDRESS_POINT_FEEDBACK_PROPERTY = "1269056830"

/**
 * Returns the corresponding constant value for a given feedback tag name.
 *
 * @param tagName the name of the feedback tag (e.g., "ADDRESS_POINT_FEEDBACK_FEATURE")
 * @return the associated constant value as a String, or an empty string if the tag is unknown
 */
def getFeedbackTagValue(tagName: String): String = tagName match {
  case "feedback:feature"  => ADDRESS_POINT_FEEDBACK_FEATURE
  case "feedback:geometry" => ADDRESS_POINT_FEEDBACK_GEOMETRY
  case tag if tag.startsWith("feedback:property") => ADDRESS_POINT_FEEDBACK_PROPERTY
  case _ => ""
}

In [0]:
import org.apache.spark.sql.functions._
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import org.locationtech.jts.geom.Geometry
import org.apache.sedona.spark.SedonaContext
import org.apache.sedona.core.spatialRDD.SpatialRDD
import scala.collection.JavaConverters._
import org.apache.spark.sql.expressions.Window
import com.tomtom.orbis.Operation
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import com.tomtom.orbis.io.spark.model.{Change, Changelet, ChangeletContext, Version, TagChange, SemanticChange, TagKey}
import com.tomtom.orbis.io.spark.model.Id
import org.apache.spark.sql.Encoders.product
import org.apache.spark.sql.functions.{col, collect_list, monotonically_increasing_id, struct}
import org.apache.spark.sql.{Dataset, SparkSession}
import com.tomtom.addressing.bulk.scala.model.GroupedChanges
import com.tomtom.orbis.io.spark.changelet.mapper.ChangeletParser
import java.io.{File, FileOutputStream}
import java.util.UUID
import scala.collection.mutable.ListBuffer
import org.apache.spark.sql.types.IntegerType



import org.apache.spark.sql.Row

// Iterate over each row in scopedApts and then over each TagKey inside the 'feedaback_tags' array
val changesDS = scopedApts.map ({ row =>
  val Product_orbis_id = row.getAs[String]("Product_orbis_id")
  val tags = row.getAs[Seq[Row]]("feedaback_tags")   // array of TagKey structs
  val tagChange = ListBuffer[TagChange]()

  tags.map { tagRow =>
      // tagKey is a struct with field 'key'
      val tagKeyStruct = tagRow.getAs[Row]("tagKey")
      val tagKey = tagKeyStruct.getAs[String]("key")
      val tagValue = tagRow.getAs[String]("value")

      println(s"$Product_orbis_id -> $tagKey : $tagValue")

      val semantic_id = getFeedbackTagValue(tagKey)

      tagChange +=TagChange.apply(
        operation = Operation.UPDATE.getNumber.toByte, 
        tagKey = TagKey.apply(tagKey), 
        value = Option.apply(tagValue),
        semanticIds = Option.empty, 
        semanticChanges = Option.apply(Seq(SemanticChange.apply(Operation.ADD.getNumber.toByte, Integer.parseUnsignedInt(semantic_id))))
      )

    }

    val idOrbis = createOrbisIdFromStringInSpark(row.getAs[String]("Product_orbis_id"))
    val change = Change.nodeChange(id = idOrbis, operation = Operation.UPDATE.getNumber.toByte, Option.empty, tagChange)

    change
  })(product[Change])
  

// println("Tag changes count: "+ changesDS.count)

// changesDS.show(50, false)

In [0]:




val withBatchId = changesDS
  .withColumn(
    "row_num",
    row_number().over(Window.orderBy(lit(1))) 
  )
  .withColumn("groupId", ((col("row_num") - 1) / 1000).cast(IntegerType))
  .drop("row_num")

val groupedChangesByBatchSize = withBatchId
      .groupBy("groupId")
      .agg(collect_list(struct("*")).alias("changes"))
      .select(col("groupId"), col("changes"))
      .as(product[GroupedChanges])

val changeletDs: Dataset[Changelet] = groupedChangesByBatchSize.map { groupedChanges =>
      Changelet(
        ChangeletContext(version = Version.apply(14533, 41894243)),
        groupedChanges.changes
      )
    }(product)

display(changeletDs)


In [0]:
changeletDs.foreach { changelet =>
      val randomSuffix = java.lang.Long.toUnsignedString(UUID.randomUUID().getLeastSignificantBits)
      val pbFilePath = f"/dbfs/mnt/opas/opas-source/palash/feedback_scoping_3/changelet_${randomSuffix}.pb"
      val file = new File(pbFilePath)
      file.getParentFile.mkdirs()
      val pbFile = new FileOutputStream(file)
      ChangeletParser.toChangelet(changelet).writeTo(pbFile)
      pbFile.close()
    }

In [0]:


println("Tag changes count: "+ changesDS.count)

changesDS.show(50, false)


changeletDs.count()